In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
from huggingface_hub import snapshot_download
import os

model_id  = "Qwen/Qwen3-0.6B"
cache_dir = os.path.expanduser("~/hf_models")

# Download only tokenizer-related files into your cache
snapshot_download(
    repo_id=model_id,
    cache_dir=cache_dir,
    local_dir_use_symlinks=False,
    allow_patterns=[
        "tokenizer.json", 
        "tokenizer.model", 
        "tokenizer_config.json",
        "vocab.json", 
        "merges.txt", 
        "special_tokens_map.json",
        "generation_config.json", 
        "*.tiktoken", 
        "*.spm"
    ],
)

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
import torch

# Remove local_files_only so it can download if not present
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    cache_dir=cache_dir,
    torch_dtype=torch.float32,   # safest for pure CPU
    low_cpu_mem_usage=True,
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [5]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [6]:
# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

In [7]:
# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

thinking content: <think>
Okay, the user wants a short introduction to a large language model. Let me start by recalling what I know about them. Large language models are AI systems designed to understand and generate human language. They're used in various applications like chatbots, translation, and content creation.

I should mention their core capabilities first. Maybe something like understanding and generating text, learning from vast data, and being adaptable. It's important to highlight their effectiveness in specific tasks. Also, the user might be looking for a concise yet comprehensive overview. I need to keep it brief but cover the key points without getting too technical. Let me check if I'm missing anything. Oh, they might be interested in how these models work or their impact. But since the request is just an introduction, focusing on the main features should suffice. Alright, time to put it all together in a friendly and informative way.
</think>
content: A large languag

In [8]:
def generate_text(prompt, temperature=1.0, top_p=0.9, top_k=0, max_length=100, repetition_penalty=1.2):
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k if top_k > 0 else None,
        repetition_penalty=repetition_penalty,
        do_sample=True
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    return content

In [9]:
generate_text("Please introduce the impact of LLM")

"A Large Language Model (LLM), such as GPT-3 or others, significantly transforms various domains by revolutionizing information processing, decision-making, and human-machine interaction. Here's a structured overview of its impactful effects:\n\n1. **Technological Advancement**:  \n   - **Efficient Communication**: LLMs enable natural language understanding and generation, improving collaboration tools (e.g., chatbots for customer service).  \n   - **Access to Knowledge**: They democratize access to vast datasets, accelerating research and discovery across fields.  \n\n2. **Societal Impact**:  \n   - **Accessibility & Inclusivity**: Models support diverse needs, helping individuals from marginalized communities gain equal opportunities.  \n   - **Education**: AI-driven tutors adapt to individual learning styles, enhancing personalized teaching experiences.  \n\n3. **Business Applications**:  \n   - **Product Innovation**: Teams leverage LLM insights to develop new products faster than 

In [10]:
import pandas as pd


# Test parameter ranges
param_ranges = {
    "temperature": [0.1, 0.5, 0.7, 1.0, 1.5, 2.0],
    "top_p": [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99],
    "top_k": [0, 5, 10, 20, 50, 100],
    "repetition_penalty": [1.0, 1.2, 1.5, 2.0],
    "max_length": [50, 100, 200]
}

# Test function
def test_generation_parameters(prompt, num_tests=3):
    results = []
    baseline_output = generate_text(prompt, temperature=1.0, top_p=1.0, top_k=0, max_length=100)
    print(f"\nBaseline output (temperature=1.0, top_p=1.0, top_k=0):\n{baseline_output}\n")
    
    # Test different parameters
    for param, values in param_ranges.items():
        for value in values:
            for i in range(num_tests):
                try:
                    # Set default parameters
                    kwargs = {
                        "temperature": 1.0,
                        "top_p": 0.9,
                        "top_k": 50,
                        "repetition_penalty": 1.2,
                        "max_length": 100
                    }
                    kwargs[param] = value  # Override the current test parameter
                    
                    # Generate text
                    output = generate_text(prompt, **kwargs)
                    print(f"params: {kwargs} \noutput: {output}")
                    
                    # Collect results
                    results.append({
                        "parameter": param,
                        "value": value,
                        "test_num": i + 1,
                        "output": output
                    })
                    print(f"Test: {param}={value} | Output: {output[:60]}...")
                
                except Exception as e:
                    print(f"Error: {param}={value} | {str(e)}")
    
    return pd.DataFrame(results)


In [11]:
df = test_generation_parameters("Please introduce the impact of LLM", 1)


Baseline output (temperature=1.0, top_p=1.0, top_k=0):
A Large Language Model (LLM)—a advanced artificial intelligence system designed to interact naturally with users—is reshaping numerous aspects of life across countless industries. Here's an overview of its key impact:

1. **Economic Shifts**:  
   - Emphasizes Personalized Efficiency: Interfaces provide features such as generating content effortlessly, streamlining workflows where possible, and enhancing productivity—all ensuring minimal effort from humans. For example, businesses leverage these tools quickly to complete complex assignments, enabling autonomous work.

2. **Rise in Communication**:  
   - Simplifies language interaction between non-native speakers or experts worldwide. Realistic conversational examples boost understanding levels significantly alongside improved decision-making abilities relying on accurate responses.

3. **Expanding Access and Participation**:
   - Enables broader integration across platforms like 

KeyboardInterrupt: 